# Growth Agent 10x Free — Colab Content Worker

This notebook runs the heavy content-generation work (LLM calls for video ideas
and scripts) on Colab's free GPU/CPU runtime, keeping everything zero-cost.

## Secrets setup (Colab Secrets — key icon in the left sidebar)

Add these four secrets before running:

| Secret name | Where to get it |
|---|---|
| `SUPABASE_URL` | Supabase dashboard → Settings → API → Project URL |
| `SUPABASE_SERVICE_KEY` | Supabase dashboard → Settings → API → service_role key |
| `GROQ_API_KEY` | console.groq.com → API Keys (free tier) |
| `GEMINI_API_KEY` | aistudio.google.com → Get API key (free tier) |

Then run all cells top to bottom. The worker pulls pending jobs from the
`job_queue` table, generates Dutch psychology video ideas + scripts, writes
drafts to the `videos` table, and marks jobs complete.

In [ ]:
!pip install -q supabase groq google-generativeai pyyaml requests

In [ ]:
import json
import datetime
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_SERVICE_KEY = userdata.get('SUPABASE_SERVICE_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

assert SUPABASE_URL, 'Missing Colab Secret: SUPABASE_URL'
assert SUPABASE_SERVICE_KEY, 'Missing Colab Secret: SUPABASE_SERVICE_KEY'

print('Secrets loaded. Groq:', bool(GROQ_API_KEY), '| Gemini:', bool(GEMINI_API_KEY))

In [ ]:
import requests

HEADERS = {
    'apikey': SUPABASE_SERVICE_KEY,
    'Authorization': f'Bearer {SUPABASE_SERVICE_KEY}',
    'Content-Type': 'application/json',
    'Prefer': 'return=representation',
}

def sb_get(table, query=''):
    r = requests.get(f'{SUPABASE_URL}/rest/v1/{table}?{query}', headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def sb_post(table, row):
    r = requests.post(f'{SUPABASE_URL}/rest/v1/{table}', headers=HEADERS,
                      data=json.dumps(row), timeout=30)
    r.raise_for_status()
    return r.json()

def sb_patch(table, match_query, patch):
    r = requests.patch(f'{SUPABASE_URL}/rest/v1/{table}?{match_query}', headers=HEADERS,
                       data=json.dumps(patch), timeout=30)
    r.raise_for_status()
    return r.json()

print('Supabase REST helpers ready.')

In [ ]:
# Pull up to 5 pending content jobs (content or colab_content)
jobs = sb_get('job_queue', 'status=eq.pending&order=created_at.asc&limit=20')
jobs = [j for j in jobs if j.get('job_type') in ('content', 'colab_content')][:5]
print(f'Found {len(jobs)} pending content job(s).')
for j in jobs:
    print(' -', j.get('id'), j.get('job_type'), (j.get('payload') or {}).get('channel_id', ''))

In [ ]:
def llm_generate(prompt, max_tokens=1500):
    """Try Groq first, fall back to Gemini. Returns generated text."""
    if GROQ_API_KEY:
        try:
            from groq import Groq
            client = Groq(api_key=GROQ_API_KEY)
            resp = client.chat.completions.create(
                model='llama-3.3-70b-versatile',
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=max_tokens,
                temperature=0.8,
            )
            return resp.choices[0].message.content
        except Exception as e:
            print('Groq failed, falling back to Gemini:', e)
    if GEMINI_API_KEY:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-2.5-flash')
        resp = model.generate_content(prompt)
        return resp.text
    raise RuntimeError('No LLM API key available (set GROQ_API_KEY or GEMINI_API_KEY).')

DUTCH_SCRIPT_PROMPT = '''Je bent een Nederlandse psychologie-contentexpert voor YouTube.
Doelgroep: Nederlandstalige kijkers geïnteresseerd in psychologie, mindset en zelfontwikkeling.
Opdracht: schrijf een volledig YouTube-script in het NEDERLANDS.
Structuur:
1. HOOK (eerste 15 seconden): een prikkelende vraag of verrassende uitspraak.
2. 3-5 HOOFDSTUKKEN: elk met een psychologisch inzicht, voorbeeld en praktische tip.
3. CTA aan het einde: abonneer + reageer met je ervaring.
Toon: warm, direct, geen jargon zonder uitleg. Lengte: ~800-1000 woorden.
Onderwerp: {topic}
Sub-niche: {sub_niche}
'''

IDEAS_PROMPT = '''Je bent een Nederlandse YouTube-strateeg voor psychologiekanalen.
Bedenk 5 virale video-ideeën in het NEDERLANDS voor de sub-niche: {sub_niche}.
Geef per idee: titel (max 60 tekens, clickbait maar eerlijk) + 1 zin beschrijving.
Nummer ze 1-5.'''

print('LLM functions ready.')

In [ ]:
generated = 0
failed = 0

for job in jobs:
    payload = job.get('payload') or {}
    channel_id = payload.get('channel_id')
    sub_niche = payload.get('sub_niche', 'psychologie')
    if not channel_id:
        print(f"Job {job.get('id')}: no channel_id, skipping.")
        continue
    try:
        ideas_text = llm_generate(IDEAS_PROMPT.format(sub_niche=sub_niche))
        # Take the first idea's title as the video topic
        first_line = next((l.strip() for l in ideas_text.splitlines() if l.strip()), 'Psychologie video')
        topic = first_line[:200]
        script = llm_generate(DUTCH_SCRIPT_PROMPT.format(topic=topic, sub_niche=sub_niche),
                             max_tokens=2500)
        title = topic.split(':', 1)[-1].strip()[:100] or topic[:100]
        sb_post('videos', {
            'channel_id': channel_id,
            'title': title,
            'script': script,
            'status': 'draft',
        })
        sb_patch('job_queue', f"id=eq.{job['id']}",
                 {'status': 'complete', 'completed_at': datetime.datetime.utcnow().isoformat()})
        generated += 1
        print(f'Job {job.get("id")}: draft created — {title[:60]}')
    except Exception as e:
        failed += 1
        try:
            sb_patch('job_queue', f"id=eq.{job['id']}", {'status': 'failed'})
        except Exception:
            pass
        print(f'Job {job.get("id")}: FAILED — {e}')

print(f'Done. generated={generated} failed={failed}')

In [ ]:
print('=== Run summary ===')
print('Jobs picked up :', len(jobs))
print('Drafts created :', generated)
print('Failed         :', failed)
try:
    pending = sb_get('job_queue', 'status=eq.pending&select=id')
    print('Still pending  :', len(pending))
except Exception as e:
    print('Could not fetch pending count:', e)

In [ ]:
%%javascript
// OPTIONAL keep-alive: prevents Colab from idling out during long runs.
// Run this cell if you want the runtime to stay connected.
function keepAlive() {
  const btn = document.querySelector('colab-connect-button');
  if (btn) btn.click();
}
setInterval(keepAlive, 60000);